<a href="https://colab.research.google.com/github/Khaledsaads/gemma-7b-sft-dpo/blob/main/gemma_7b_sft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -- no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-rippde6f/unsloth_550afc58f8414eb9a0ed8742360f5629
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-rippde6f/unsloth_550afc58f8414eb9a0ed8742360f5629
  Resolved https://github.com/unslothai/unsloth.git to commit 12eb64b867dd3db31f76d95d80b33e955f443600
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 34.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 65.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 121.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 46.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 139.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 147.4 MB/s  0:00:00
  Created wheel for uns

In [ ]:
! pip install trl

In [ ]:
! pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install -U torchao

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import AutoTokenizer
from unsloth import FastLanguageModel
from datasets import load_dataset

/usr/local/lib/python3.13/dist-packages/unsloth/__init__.py:1678: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
tokenizer = AutoTokenizer.from_pretrained("unsloth/gemma-7b-bnb-4bit")

In [ ]:
dataset = load_dataset("HuggingFaceH4/deita-10k-v0-sft")

In [ ]:
dataset

DatasetDict({
    train_sft: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 9500
    })
    test_sft: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 500
    })
    train_gen: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 9500
    })
    test_gen: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 500
    })
})

In [ ]:
train_data = dataset['train_sft']

In [ ]:
import pandas as pd
train_sft = pd.DataFrame(train_data)
train_sft.head(3)

,prompt,prompt_id,messages
0,make a snakemake workflow starting with whole ...,635d9cbab07b3366bcd3e266b079df795dcb73039a55ec...,[{'content': 'make a snakemake workflow starti...
1,Hello,185f8db32271fe25f561a6fc938b2e264306ec304eda51...,"[{'content': 'Hello', 'role': 'user'}, {'conte..."
2,Now you are a programming assistant. Reply wit...,99255b3542fd88816893be2eaaba68bcd79714fc73b7a3...,[{'content': 'Now you are a programming assist...


In [ ]:
train_sft.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9500 entries, 0 to 9499
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   prompt     9500 non-null   object
 1   prompt_id  9500 non-null   object
 2   messages   9500 non-null   object
dtypes: object(3)
memory usage: 222.8+ KB


In [ ]:
print(train_sft.iloc[0]["prompt"])

make a snakemake workflow starting with whole genome paired end fastq files and produce hg19 aligned bam files per GATK best practices please


In [ ]:
train_data.features

{'prompt': Value('string'),
 'prompt_id': Value('string'),
 'messages': List({'content': Value('string'), 'role': Value('string')})}

In [ ]:
val_data = dataset["test_sft"]

In [ ]:
m =0
for message in train_sft.iloc[0]["messages"]:
  m+=1
  print(message["role"])
  print(message["content"])


user
make a snakemake workflow starting with whole genome paired end fastq files and produce hg19 aligned bam files per GATK best practices please
assistant
Sure, I can help you create a Snakemake workflow for aligning paired-end fastq files to the hg19 reference genome and producing aligned bam files following GATK best practices. Here's a basic workflow you can use as a starting point:
```python
# Define input and output file paths
fastq_dir = "path/to/fastq/files"
reference_genome = "path/to/hg19.fa"
output_dir = "path/to/output/directory"

# Define the rule to align paired-end reads using BWA
rule align_reads:
    input:
        R1 = fastq_dir + "/{sample}_R1.fastq",
        R2 = fastq_dir + "/{sample}_R2.fastq",
        index = reference_genome + ".bwt"
    output:
        bam = output_dir + "/{sample}.bam",
        log = output_dir + "/{sample}.bwa.log"
    shell:
        "bwa mem -M -t 4 -R '@RG\\tID:{wildcards.sample}\\tSM:{wildcards.sample}\\tPL:illumina' {input.index} {input.

In [ ]:
print(f"Number of user turns in first conversation = {int(m/2)}")

Number of user turns in first conversation = 11


In [ ]:
import json
duplicated_rows = train_sft["messages"].apply(lambda x : json.dumps(x))
print(f"Duplicate conversations found: {duplicated_rows.duplicated().sum()}")

Duplicate conversations found: 0


In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = 'gemma'
)

In [ ]:
text = tokenizer.apply_chat_template(
    train_sft.iloc[0]["messages"],
    tokenize = False,
    add_generation_prompt =False
)
print(text)

<bos><start_of_turn>user
make a snakemake workflow starting with whole genome paired end fastq files and produce hg19 aligned bam files per GATK best practices please<end_of_turn>
<start_of_turn>model
Sure, I can help you create a Snakemake workflow for aligning paired-end fastq files to the hg19 reference genome and producing aligned bam files following GATK best practices. Here's a basic workflow you can use as a starting point:
```python
# Define input and output file paths
fastq_dir = "path/to/fastq/files"
reference_genome = "path/to/hg19.fa"
output_dir = "path/to/output/directory"

# Define the rule to align paired-end reads using BWA
rule align_reads:
    input:
        R1 = fastq_dir + "/{sample}_R1.fastq",
        R2 = fastq_dir + "/{sample}_R2.fastq",
        index = reference_genome + ".bwt"
    output:
        bam = output_dir + "/{sample}.bam",
        log = output_dir + "/{sample}.bwa.log"
    shell:
        "bwa mem -M -t 4 -R '@RG\\tID:{wildcards.sample}\\tSM:{wildcards.

In [ ]:
# import numpy as np

# def count(batch):
#     texts = [tokenizer.apply_chat_template(m, tokenize=False) for m in batch["messages"]]
#     return {"n": [len(x) for x in tokenizer(texts, add_special_tokens=False)["input_ids"]]}

# lengths = np.array(train_data.map(count, batched=True, remove_columns=train_data.column_names)["n"])
# for p in (50, 90, 95, 99):
#     print(p, int(np.percentile(lengths, p)))
# print("max", lengths.max())

In [ ]:
new_train = train_data.select_columns("messages")
new_val = val_data.select_columns("messages")

In [ ]:
print(f"new train features ={new_train.features}")
print(f"new val features ={new_val.features}")

new train features ={'messages': List({'content': Value('string'), 'role': Value('string')})}
new val features ={'messages': List({'content': Value('string'), 'role': Value('string')})}


In [ ]:
print(f"new train length ={len(new_train)}")
print(f"new val length ={len(new_val)}")

new train length =9500
new val length =500


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-7b-bnb-4bit",
    dtype = None,
    max_seq_length = 4096,
    load_in_4bit = True
)

==((====))==  Unsloth 2026.10.7: Fast Gemma patching. Transformers: 5.17.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.0. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-7b-bnb-4bit as a legacy tokenizer.


In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = 'gemma'
)

Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_u1haggz5/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/_unsloth_sentencepiece_temp/tokenizer_u1haggz5.


In [ ]:
model

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): GemmaTextScaledWordEmbedding(256000, 3072, padding_idx=0)
    (layers): ModuleList(
      (0-27): 28 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear4bit(in_features=3072, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=3072, out_features=4096, bias=False)
          (v_proj): Linear4bit(in_features=3072, out_features=4096, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=3072, bias=False)
          (rotary_emb): GemmaFixedRotaryEmbedding()
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear4bit(in_features=3072, out_features=24576, bias=False)
          (up_proj): Linear4bit(in_features=3072, out_features=24576, bias=False)
          (down_proj): Linear4bit(in_features=24576, out_features=3072, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((3072,), eps=1e-06)
        (post_at

In [ ]:
peft_model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing= False,
    max_seq_length = 4096,
    random_state= 3407,
    use_rslora = False
)

Unsloth 2026.10.7 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (GemmaDecoderLayer).


In [ ]:
(10000/ 16 )

625.0

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer_args = SFTConfig(
    output_dir = "output",
    num_train_epochs = 2,
    per_device_train_batch_size =1,
    gradient_accumulation_steps=16,
    eval_strategy = "steps",
    save_strategy = "steps",
    save_steps = 140,
    eval_steps = 140,
    logging_steps =60,
    warmup_steps = 60,
    learning_rate = 2e-4,
    optim = "paged_adamw_8bit",
    lr_scheduler_type = "cosine",
    weight_decay = 0.05,
    gradient_checkpointing =False,
    bf16=True,
    fp16= False,
    max_length = 4096,
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better  = False,
)

In [ ]:
from transformers import EarlyStoppingCallback
trainer = SFTTrainer(
    peft_model,
    train_dataset = new_train,
    eval_dataset = new_val,
    args = trainer_args,
    processing_class= tokenizer,
    callbacks = [EarlyStoppingCallback(early_stopping_patience = 3)]
)

Unsloth: Tokenizing ["messages"] (num_proc=6):   0%|          | 0/9500 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["messages"] (num_proc=6):   0%|          | 0/500 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [ ]:
print(tokenizer.decode(trainer.train_dataset[0]["input_ids"][:60]))

<bos><start_of_turn>user
make a snakemake workflow starting with whole genome paired end fastq files and produce hg19 aligned bam files per GATK best practices please<end_of_turn>
<start_of_turn>model
Sure, I can help you create a Snakemake workflow for aligning paired-end fastq files to the hg1


In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 9,500 | Num Epochs = 2 | Total steps = 1,188
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 16
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 16 x 1) = 16
 "-____-"     Trainable parameters = 50,003,968 of 8,587,684,864 (0.58% trained)


Step,Training Loss,Validation Loss
140,0.891226,0.879116
280,0.845567,0.863987
420,0.848904,0.861475
560,0.834984,0.856072
700,0.759533,0.858107
840,0.765465,0.852976
980,0.726226,0.852122


Unsloth: Not an error, but `num_items_in_batch` could not be counted for GemmaForCausalLM's labels, so each micro-batch loss is a mean.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
Unsloth: Restored added_tokens_decoder metadata in output/checkpoint-140/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in output/checkpoint-140.
Unsloth: Restored added_tokens_decoder metadata in output/checkpoint-280/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in output/checkpoint-280.
Unsloth: Restored added_tokens_decoder metadata in output/checkpoint-420/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in output/checkpoint-420.
Unsloth: Restored added_tokens_decoder metadata in output/checkpoint-560/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in output/checkpoint-560.
Un

Step,Training Loss,Validation Loss
140,0.891226,0.879116
280,0.845567,0.863987
420,0.848904,0.861475
560,0.834984,0.856072
700,0.759533,0.858107
840,0.765465,0.852976
980,0.726226,0.852122
1120,0.752558,0.850602
1188,0.737079,0.850645


Unsloth: Restored added_tokens_decoder metadata in output/checkpoint-1120/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in output/checkpoint-1120.
Unsloth: Restored added_tokens_decoder metadata in output/checkpoint-1188/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in output/checkpoint-1188.


TrainOutput(global_step=1188, training_loss=0.8043529433433456, metrics={'train_runtime': 16377.3661, 'train_samples_per_second': 1.16, 'train_steps_per_second': 0.073, 'total_flos': 3.2472454796604457e+18, 'train_loss': 0.8043529433433456, 'epoch': 2.0})

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
repo_id = "Khaledsaad1/gemma-7b-deita-10k-qlora"
api.create_repo(
    repo_id = repo_id,
    repo_type = "model" ,
    exist_ok =True

    )
api.upload_folder(
    repo_id = repo_id,
    repo_type = "model",
    folder_path = "/content/output/checkpoint-1120"
)

CommitInfo(commit_url='https://huggingface.co/Khaledsaad1/gemma-7b-deita-10k-qlora/commit/f2a6688cb81f75875a3f7e96e6ee439f81c9654b', commit_message='Upload folder using huggingface_hub', commit_description='', oid='f2a6688cb81f75875a3f7e96e6ee439f81c9654b', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Khaledsaad1/gemma-7b-deita-10k-qlora', endpoint='https://huggingface.co', repo_type='model', repo_id='Khaledsaad1/gemma-7b-deita-10k-qlora'), pr_revision=None, pr_num=None)

In [ ]:
tokenizer.push_to_hub(repo_id)

README.md:   0%|          | 0.00/5.23k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpwakkw467/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /tmp/tmpwakkw467.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpwakkw467/tokenizer.json:  47%|####6     | 16.0MB / 34.4MB            

  ...pwakkw467/tokenizer.model: 100%|##########| 4.24MB / 4.24MB            

Validating                    :           |   0%            

No files have been modified since last commit. Skipping to prevent empty commit.


In [5]:
benchmark_examples= [
  [
    { "role": "user", "content": "Write a Python function to check if a string is a palindrome, ignoring spaces and punctuation." },
    { "role": "assistant", "content": "def is_palindrome(s: str) -> bool:\n    cleaned = ''.join(ch.lower() for ch in s if ch.isalnum())\n    return cleaned == cleaned[::-1]" }
  ],
  [
    { "role": "user", "content": "Calculate the compound interest for a principal of $1,000 at an annual rate of 5% compounded annually for 3 years." },
    { "role": "assistant", "content": "The formula for compound interest is $A = P(1 + r/n)^{nt}$. \nHere, $P = 1000$, $r = 0.05$, $n = 1$, and $t = 3$.\n$A = 1000 \times (1 + 0.05)^3 = 1000 \times 1.157625 = 1157.63$.\nThus, the total amount is $1,157.63, yielding $157.63 in interest." }
  ],
  [
    { "role": "user", "content": "Provide a brief overview of the Gutenberg printing press and its impact on history." },
    { "role": "assistant", "content": "Invented by Johannes Gutenberg around 1440, the printing press introduced movable metal type to Europe. It revolutionized communication by mass-producing books, dramatically lowering costs, increasing literacy rates, and accelerating the spread of scientific and religious knowledge." }
  ],
  [
    { "role": "user", "content": "What is the difference between supervised and unsupervised machine learning?" },
    { "role": "assistant", "content": "Supervised learning uses labeled datasets to train models to predict outcomes or classify data accurately. Unsupervised learning works with unlabeled data, allowing the algorithm to discover hidden patterns, groupings (clustering), or underlying structures on its own." }
  ],
  [
    { "role": "user", "content": "Give me a quick Bash command to find all files larger than 100MB in the current directory." },
    { "role": "assistant", "content": "You can use the `find` command:\n`find . -type f -size +100M`" }
  ],
  [
    { "role": "user", "content": "Write a SQL query to find the top 3 highest-paid employees from an 'employees' table." },
    { "role": "assistant", "content": "SELECT name, salary FROM employees ORDER BY salary DESC LIMIT 3;" }
  ],
  [
    { "role": "user", "content": "Explain the concept of 'opportunity cost' in economics with a simple example." },
    { "role": "assistant", "content": "Opportunity cost is the potential benefit given up when choosing one alternative over another. For example, if you spend $10 on a movie ticket, the opportunity cost is whatever else you could have bought or enjoyed with that $10, like a book or a meal." }
  ],
  [
    { "role": "user", "content": "What are the primary macronutrients required by the human body?" },
    { "role": "assistant", "content": "The three primary macronutrients are carbohydrates (the body's main energy source), proteins (essential for tissue repair and growth), and fats (vital for hormone regulation and energy storage)." }
  ],
  [
    { "role": "user", "content": "Write a short JavaScript snippet to toggle a CSS class on an element when clicked." },
    { "role": "assistant", "content": "const button = document.querySelector('#myButton');\nbutton.addEventListener('click', () => {\n  document.body.classList.toggle('dark-mode');\n});" }
  ],
  [
    { "role": "user", "content": "What causes the phases of the Moon as seen from Earth?" },
    { "role": "assistant", "content": "The phases of the Moon are caused by its changing position relative to the Earth and the Sun as it orbits our planet. This changes the angle at which sunlight hits the lunar surface visible from Earth." }
  ],
  [
    { "role": "user", "content": "Provide a helpful tip for managing time effectively using the Pomodoro Technique." },
    { "role": "assistant", "content": "Work with intense focus for 25 minutes without distractions, then take a mandatory 5-minute break. After completing four cycles, take a longer break of 15 to 30 minutes to maintain sustainable productivity." }
  ],
  [
    { "role": "user", "content": "What is a RESTful API and what are its key HTTP methods?" },
    { "role": "assistant", "content": "A RESTful API is an architectural style for designing networked applications using HTTP requests. Key methods include GET (retrieve data), POST (create data), PUT/PATCH (update data), and DELETE (remove data)." }
  ],
  [
    { "role": "user", "content": "How do you declare a constant variable in modern JavaScript?" },
    { "role": "assistant", "content": "You use the `const` keyword. For example:\n`const API_URL = 'https://api.example.com';`\nNote that `const` prevents re-assignment of the variable identifier, though mutable objects assigned to it can still have their properties modified." }
  ],
  [
    { "role": "user", "content": "Define 'recursion' in computer science simply." },
    { "role": "assistant", "content": "Recursion is a programming technique where a function calls itself to solve smaller sub-instances of the same problem until it reaches a base case that stops the recursion." }
  ],
  [
    { "role": "user", "content": "What is the capital of Australia, and why is it often confused?" },
    { "role": "assistant", "content": "The capital of Australia is Canberra. It is frequently confused with larger coastal cities like Sydney or Melbourne because those are major global and cultural hubs." }
  ],
  [
    { "role": "user", "content": "Write a CSS rule to center a flexbox item both horizontally and vertically." },
    { "role": "assistant", "content": ".container {\n  display: flex;\n  justify-content: center;\n  align-items: center;\n}" }
  ],
  [
    { "role": "user", "content": "What is the difference between a process and a thread in operating systems?" },
    { "role": "assistant", "content": "A process is an independent execution unit with its own memory space and system resources. A thread is a smaller unit of execution within a process, sharing the same memory space to allow for faster context switching and communication." }
  ],
  [
    { "role": "user", "content": "Give me a quick tip on how to structure a professional cover letter." },
    { "role": "assistant", "content": "Keep it to one page. Structure it with an engaging opening stating the role you want, 1-2 body paragraphs highlighting specific achievements matching their needs, and a strong closing call to action expressing enthusiasm." }
  ],
  [
    { "role": "user", "content": "How does photosynthesis convert light energy into chemical energy?" },
    { "role": "assistant", "content": "During photosynthesis, plants capture sunlight using chlorophyll in chloroplasts. They use this solar energy to convert carbon dioxide from the air and water from the soil into glucose (sugar) for food, releasing oxygen as a byproduct." }
  ],
  [
    { "role": "user", "content": "What is Git rebase and how does it differ from a merge?" },
    { "role": "assistant", "content": "Git merge combines branches by creating a new 'merge commit,' preserving the exact historical timeline. Git rebase moves or combines a sequence of commits to a new base commit, resulting in a clean, linear commit history." }
  ],
  [
    { "role": "user", "content": "What is the function of mitochondria in human cells?" },
    { "role": "assistant", "content": "Mitochondria are known as the powerhouses of the cell because they generate most of the chemical energy (ATP) needed to power biochemical reactions via cellular respiration." }
  ],
  [
    { "role": "user", "content": "Write a regular expression to match a standard email address format." },
    { "role": "assistant", "content": "^[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\\.[a-zA-Z0-9-.]+$" }
  ],
  [
    { "role": "user", "content": "What is inflation and what typically causes it?" },
    { "role": "assistant", "content": "Inflation is the general increase in prices and fall in the purchasing value of money over time. It is typically caused by demand-pull factors (high consumer demand outpacing supply), cost-push factors (increased production costs), or excessive money supply growth." }
  ],
  [
    { "role": "user", "content": "Explain what an index does in a relational database." },
    { "role": "assistant", "content": "A database index is a data structure (like a B-tree) that improves the speed of data retrieval operations on a table at the cost of additional storage space and slower write performance." }
  ],
  [
    { "role": "user", "content": "What are the core pillars of Object-Oriented Programming (OOP)?" },
    { "role": "assistant", "content": "The four core pillars are Encapsulation (bundling data and methods), Inheritance (reusing code across classes), Polymorphism (using a unified interface for different data types), and Abstraction (hiding complex implementation details)." }
  ],
  [
    { "role": "user", "content": "How can you check open ports on a Linux server?" },
    { "role": "assistant", "content": "You can use the `ss` command or netstat:\n`ss -tuln`\nThis displays all listening TCP and UDP ports numerically." }
  ],
  [
    { "role": "user", "content": "What is the Heisenberg Uncertainty Principle in quantum mechanics?" },
    { "role": "assistant", "content": "It states that you cannot simultaneously know both the precise position and momentum of a particle; the more accurately one is measured, the less precisely the other can be known." }
  ],
  [
    { "role": "user", "content": "Write a Python snippet to read a CSV file using the built-in module." },
    { "role": "assistant", "content": "import csv\n\nwith open('data.csv', mode='r') as file:\n    reader = csv.reader(file)\n    for row in reader:\n        print(row)" }
  ],
  [
    { "role": "user", "content": "What is the difference between symmetric and asymmetric encryption?" },
    { "role": "assistant", "content": "Symmetric encryption uses a single secret key for both encryption and decryption, making it fast but requiring secure key sharing. Asymmetric encryption uses a key pair (a public key for encryption and a private key for decryption), enhancing security for key exchange." }
  ],
  [
    { "role": "user", "content": "Give a quick summary of the Pythagorean theorem." },
    { "role": "assistant", "content": "For any right-angled triangle, the square of the length of the hypotenuse ($c$) equals the sum of the squares of the lengths of the other two sides ($a$ and $b$): $a^2 + b^2 = c^2$." }
  ],
  [
    { "role": "user", "content": "What is a Docker container?" },
    { "role": "assistant", "content": "A Docker container is a lightweight, standalone, executable package of software that includes everything needed to run an application—code, runtime, system tools, libraries, and settings—ensuring consistent execution across environments." }
  ],
  [
    { "role": "user", "content": "What are the primary layers of the OSI model?" },
    { "role": "assistant", "content": "The 7 layers from bottom to top are: Physical, Data Link, Network, Transport, Session, Presentation, and Application (remembered often as 'Please Do Not Throw Sausage Pizza Away')." }
  ],
  [
    { "role": "user", "content": "How do you handle asynchronous operations in modern JavaScript?" },
    { "role": "assistant", "content": "Modern JavaScript handles asynchronous operations using `Promises` and `async/await` syntax, which allows writing asynchronous code that looks and behaves like synchronous code." }
  ],
  [
    { "role": "user", "content": "What is the significance of the pH scale in chemistry?" },
    { "role": "assistant", "content": "The pH scale measures how acidic or basic a watery solution is, ranging from 0 to 14. A value of 7 is neutral, values below 7 indicate acidity, and values above 7 indicate alkalinity (basicity)." }
  ],
  [
    { "role": "user", "content": "Provide a simple definition of cloud computing." },
    { "role": "assistant", "content": "Cloud computing is the delivery of computing services—including servers, storage, databases, networking, and software—over the internet ('the cloud') rather than relying on local hardware." }
  ],
  [
    { "role": "user", "content": "Write a simple function in Python to calculate the factorial of a number iteratively." },
    { "role": "assistant", "content": "def factorial(n):\n    result = 1\n    for i in range(2, n + 1):\n        result *= i\n    return result" }
  ],
  [
    { "role": "user", "content": "What is a deadlock in concurrent computing?" },
    { "role": "assistant", "content": "A deadlock is a situation where two or more competing actions are each waiting for the other to finish, resulting in none of them ever proceeding." }
  ],
  [
    { "role": "user", "content": "What is the difference between `let`, `const`, and `var` in JavaScript?" },
    { "role": "assistant", "content": "`var` is function-scoped and allows redeclaration. `let` is block-scoped and allows reassignment but not redeclaration. `const` is block-scoped, prevents reassignment, and requires immediate initialization." }
  ],
  [
    { "role": "user", "content": "Explain how DNS (Domain Name System) works." },
    { "role": "assistant", "content": "DNS acts as the phonebook of the internet, translating human-friendly domain names (like example.com) into machine-readable IP addresses (like 192.0.2.1) through a hierarchical network of name servers." }
  ],
  [
    { "role": "user", "content": "What is a closure in programming languages like JavaScript?" },
    { "role": "assistant", "content": "A closure is a function bundled together with references to its lexical scope environment, allowing an inner function to access variables from an outer function even after the outer function has finished executing." }
  ],
  [
    { "role": "user", "content": "What are the main goals of database normalization?" },
    { "role": "assistant", "content": "Database normalization aims to minimize data redundancy, eliminate undesirable characteristics like insertion, update, and deletion anomalies, and ensure data dependencies make logical sense." }
  ],
  [
    { "role": "user", "content": "What is the difference between HTTP and HTTPS?" },
    { "role": "assistant", "content": "HTTPS (Hypertext Transfer Protocol Secure) is the secure version of HTTP. It uses SSL/TLS encryption to protect data transmitted between a web browser and a website from eavesdropping or tampering." }
  ],
  [
    { "role": "user", "content": "Write a Python script snippet to fetch JSON data from a public API." },
    { "role": "assistant", "content": "import requests\n\nresponse = requests.get('https://api.github.com')\nif response.status_code == 200:\n    data = response.json()\n    print(data)" }
  ],
  [
    { "role": "user", "content": "What is polymorphism in object-oriented programming?" },
    { "role": "assistant", "content": "Polymorphism allows different classes to share the same method interface while implementing behavior specific to their own class, enabling objects to be treated interchangeably through a common parent type." }
  ],
  [
    { "role": "user", "content": "What is the primary purpose of a firewall in network security?" },
    { "role": "assistant", "content": "A firewall monitors and filters incoming and outgoing network traffic based on predetermined security rules, acting as a barrier between a trusted internal network and untrusted external networks like the internet." }
  ],
  [
    { "role": "user", "content": "Give a quick summary of what Agile software development is." },
    { "role": "assistant", "content": "Agile is an iterative approach to project management and software development that helps teams deliver value to customers faster through small, frequent cycles (sprints) rather than a monolithic sequential release plan." }
  ],
  [
    { "role": "user", "content": "What is a hash function and what are its key properties?" },
    { "role": "assistant", "content": "A hash function maps data of arbitrary size to fixed-size values. Key properties include determinism (same input always yields same output), quick computation, preimage resistance (hard to reverse), and avalanche effect (small input change drastically alters output)." }
  ],
  [
    { "role": "user", "content": "What is the difference between a primary key and a foreign key?" },
    { "role": "assistant", "content": "A primary key uniquely identifies each record in a table and cannot contain null values. A foreign key is a field (or collection of fields) in one table that uniquely references a primary key in another table, establishing a relationship between them." }
  ],
  [
    { "role": "user", "content": "What is an algorithm's Big O notation used for?" },
    { "role": "assistant", "content": "Big O notation describes the limiting behavior or complexity of an algorithm, classifying how its execution time or memory footprint scales relative to the input size ($n$) as $n$ grows large." }
  ],
  [
    { "role": "user", "content": "Briefly explain the MVC (Model-View-Controller) architectural pattern." },
    { "role": "assistant", "content": "MVC divides an application into three interconnected parts: the Model manages data and business logic; the View handles UI display and rendering; and the Controller accepts user input, manipulates the Model, and returns the appropriate View." }
  ]
]

In [3]:
from huggingface_hub import login
login()

In [4]:
from unsloth import FastLanguageModel
repo_id = "Khaledsaad1/gemma-7b-deita-10k-qlora"
loaded_model, loaded_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Khaledsaad1/gemma-7b-deita-10k-qlora",
    dtype =None,
    max_seq_length =4096,
    load_in_4bit = True
)
FastLanguageModel.for_inference(loaded_model)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.10.8: Fast Gemma patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/gemma-7b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.10.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (GemmaDecoderLayer).


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GemmaForCausalLM(
      (model): GemmaModel(
        (embed_tokens): GemmaTextScaledWordEmbedding(256000, 3072, padding_idx=0)
        (layers): ModuleList(
          (0-27): 28 x GemmaDecoderLayer(
            (self_attn): GemmaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
             

In [6]:
from unsloth.chat_templates import get_chat_template
from transformers import AutoTokenizer
tokenizer = get_chat_template(
    loaded_tokenizer,
    chat_template = "gemma")

In [7]:
import torch
text = [{"role": "user", "content": "write a python code that check word is palindrome or not"}]
input_ids_list = []
for example in benchmark_examples:
  input_ids = tokenizer.apply_chat_template(
      [example[0]],
      tokenize = True,
      add_generation_prompt = True,
      return_tensors= "pt").to("cuda")
  input_ids_list.append(input_ids)


In [15]:
import json
FastLanguageModel.for_inference(loaded_model)
from tqdm import tqdm
loaded_model.eval()
results = []

for input_ids in tqdm(input_ids_list):
  with torch.inference_mode():
    output = loaded_model.generate(
        input_ids = input_ids,
        do_sample=True,
        top_p = 0.9,
        top_k = 50,
        temperature = 0.7,
        max_tokens = None,
        use_cache = True
    )
    response =tokenizer.decode(output[0], skip_special_tokens=True)

  sample = []
  sample.append({
      "role": "user",
      "content": tokenizer.decode(
          output[0, 4:input_ids.shape[-1]- 2],
          skip_special_tokens = True
          )
      })
  sample.append({
      "role": "assistant",
      "content": tokenizer.decode(
          output[0, input_ids.shape[-1]:],
          skip_special_tokens = True
          )
      })
  results.append(sample)
with open("test_cases-sft.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)


100%|██████████| 50/50 [10:43<00:00, 12.88s/it]


In [16]:
i = 0
for sample in results:
  for role in sample:
    print(role["role"])
    print(role["content"])
  print("#"*100)
  i+=1
  if i > 15:
    break

user
Write a Python function to check if a string is a palindrome, ignoring spaces and punctuation.

assistant
Here's a Python function that checks if a string is a palindrome, ignoring spaces and punctuation:
```python
def is_palindrome(s):
    s = s.lower().replace(" ", "").replace(".", "").replace(",", "").replace("!", "").replace("?", "").replace("'", "").replace("(", "").replace(")", "")
    return s == s[::-1]
```
The function first removes all spaces and punctuation from the input string using the `replace()` method. Then, it compares the original string to its reverse using the `[::-1]` notation. If the two strings are the same, the function returns `True`; otherwise, it returns `False`.
####################################################################################################
user
Calculate the compound interest for a principal of $1,000 at an annual rate of 5% compounded annually for 3 years.

assistant
The formula for compound interest is:

A = P \* (1 + r/n)^(n\*t